# 🌸 ShiroLLM - AI Chatbot (Google Colab Pro Edition)
Jalankan AI Shiro menggunakan akselerasi GPU di Google Colab (T4 / L4 / A100)!

### Fitur Unggulan:
- ⚡ **Instalasi Kilat**: Menggunakan pre-built CUDA wheel (selesai dalam 5-10 detik, tanpa compile lama)
- 🚀 **Akselerasi GPU CUDA Penuh**: Respon AI sangat cepat (< 0.8 - 1.5 detik)
- 🧠 **Model Anti-Halusinasi & Cerdas**: Qwen2.5-7B (Cepat & Cerdas) atau Qwen2.5-14B (Genius Level untuk Colab Pro)
- 💡 **Continuous In-Context Self-Learning**: Shiro belajar dan meniru gaya interaksi dari percakapan sebelumnya secara otomatis
- 🎓 **Fitur Latih Mandiri (Fine-Tuning)**: Ekspor riwayat obrolan untuk fine-tuning LoRA / Unsloth langsung di Colab
- 🌐 **Web UI Publik**: Bisa diakses via Ngrok atau Cloudflare Tunnel langsung dari HP/PC Anda!

## Langkah 1: Cek & Verifikasi GPU
Jalankan cell ini untuk memastikan Colab Anda menggunakan GPU (T4 atau A100).

In [ ]:
import subprocess
res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if res.returncode == 0:
    print("=" * 60)
    print("🚀 GPU AKTIF! Siap untuk inferensi super cepat (< 1-2 detik)")
    print("=" * 60)
    print(res.stdout)
else:
    print("=" * 60)
    print("❌ PERINGATAN: GPU BELUM DIAKTIFKAN!")
    print("Colab Anda saat ini masih mode CPU biasa. Respon AI akan SANGAT LAMBAT (1-2 menit)!")
    print("=" * 60)
    print("👉 CARA AKTIFKAN GPU SEKARANG (HANYA 3 KLIK):")
    print("1. Klik menu 'Runtime' di atas (atau klik tombol 'Change runtime type' di kanan bawah)")
    print("2. Pada Hardware accelerator, pilih 'T4 GPU' (atau 'A100 GPU' jika punya Colab Pro)")
    print("3. Klik 'Save'")
    print("=" * 60)


## Langkah 2: Clone Repository

In [ ]:
%cd /content
!rm -rf ShiroLLM-AI-CHAT
!git clone https://github.com/Renn-devBI/ShiroLLM-AI-CHAT.git
%cd /content/ShiroLLM-AI-CHAT

## Langkah 3: Install Dependencies (Instan / 5-10 Detik ⚡)
Menggunakan pre-compiled CUDA wheel resmi sehingga **TIDAK PERLU COMPILE 15 MENIT**!

In [ ]:
# 1. Install dependencies web, pyngrok, & CUDA runtime libraries (libcudart)
!pip install -r requirements.txt

# 2. Install pre-compiled llama-cpp-python CUDA wheel (Hanya 5 detik!)
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 || pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

## Langkah 4: Download Model Tercanggih (Qwen 2.5 7B / 14B) ⭐
Pilih model yang Anda inginkan:
- `"7b"`: **Qwen2.5-7B-Instruct** (~5.4 GB, Sangat fasih Bahasa Indonesia, anti-halusinasi, super cepat di T4 GPU)
- `"14b"`: **Qwen2.5-14B-Instruct** (~9.0 GB, Genius level, penalaran sangat mendalam, cocok untuk Colab Pro T4/L4/A100)

In [ ]:
import os
# Ubah ke "14b" jika ingin model genius di Colab Pro, atau "7b" untuk kecepatan maksimal
os.environ["MODEL_CHOICE"] = "7b"

!python download_model.py

## Langkah 5 (Opsi A - NGROK): Jalankan Server dengan Ngrok ⭐
Gunakan opsi ini jika Anda memiliki token Ngrok (gratis di [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)).

In [ ]:
import os
import glob
from pyngrok import ngrok
import getpass

# Pastikan libcudart.so.12 terdeteksi di LD_LIBRARY_PATH
cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib') + \
            glob.glob('/usr/lib/python*/dist-packages/nvidia/*/lib') + \
            ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
valid_paths = [p for p in cuda_libs if os.path.isdir(p)]
if valid_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(valid_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Masukkan token Anda di sini (atau ketik saat diminta)
NGROK_TOKEN = ""  # <-- Masukkan token Anda di dalam tanda kutip jika mau praktis

if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("Masukkan Ngrok Authtoken Anda (dari dashboard.ngrok.com): ")

ngrok.set_auth_token(NGROK_TOKEN.strip())
ngrok.kill() # Tutup tunnel lama jika ada
public_url = ngrok.connect(7474)

print("\n" + "="*60)
print(f"🎉 BUKA LINK WEB UI SHIRO DI SINI:")
print(f"👉 {public_url}")
print("="*60 + "\n")

# Set GPU layers ke -1 (offload semua layer model ke GPU VRAM Colab)
os.environ["N_GPU_LAYERS"] = "-1"

# Jalankan server web Shiro
!python web.py

## Langkah 5 (Opsi B - CLOUDFLARE): Jalankan Tanpa Perlu Akun / Token ⚡
Gunakan opsi ini jika ingin langsung membuka link tanpa perlu login / token Ngrok!

In [ ]:
import os
import glob
import subprocess
import threading
import time
import re

# Pastikan libcudart.so.12 terdeteksi di LD_LIBRARY_PATH
cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib') + \
            glob.glob('/usr/lib/python*/dist-packages/nvidia/*/lib') + \
            ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
valid_paths = [p for p in cuda_libs if os.path.isdir(p)]
if valid_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(valid_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Set GPU layers ke -1
os.environ["N_GPU_LAYERS"] = "-1"

# Download Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# Jalankan tunnel dan simpan log ke file
!rm -f /content/tunnel.log
tunnel_proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7474"], 
                               stdout=open("/content/tunnel.log", "w"), 
                               stderr=subprocess.STDOUT)

# Monitor file log sampai link https://*.trycloudflare.com muncul
def wait_for_url():
    for _ in range(30):
        time.sleep(1)
        if os.path.exists("/content/tunnel.log"):
            with open("/content/tunnel.log", "r") as f:
                content = f.read()
                match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
                if match:
                    print("\n" + "="*60)
                    print(f"🎉 BUKA LINK WEB UI SHIRO DI SINI:")
                    print(f"👉 {match.group(1)}")
                    print("="*60 + "\n")
                    return
    print("\n⚠️ Link belum terdeteksi otomatis, silakan cek isi /content/tunnel.log")

threading.Thread(target=wait_for_url, daemon=True).start()

# Jalankan server web Shiro
!python web.py

## (Opsional) Sinkronisasi / Backup Memori ke Google Drive
Jalankan cell ini jika ingin menyimpan ingatan Shiro ke Google Drive agar tidak hilang saat runtime Colab di-restart.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')
backup_dir = '/content/drive/MyDrive/Shiro_Memory_Backup'
os.makedirs(backup_dir, exist_ok=True)

# Backup file ingatan saat ini ke Google Drive
if os.path.exists('ingatan_shiro.json'):
    shutil.copy('ingatan_shiro.json', os.path.join(backup_dir, 'ingatan_shiro.json'))
    print(f"✓ Ingatan Shiro berhasil dibackup ke: {backup_dir}")


## Langkah 6: 🎓 Latih / Fine-Tune Shiro dari Percakapan Sebelumnya
Jalankan cell ini untuk mengekstrak seluruh obrolan dan ingatan Shiro menjadi dataset training standar (ChatML, Alpaca, ShareGPT).
Dataset ini bisa langsung dipakai untuk melatih (fine-tune) model menggunakan Unsloth / LoRA di Colab Pro!

In [ ]:
# Ekspor ingatan dan percakapan Shiro ke format training
!python export_training_data.py

# Tampilkan cuplikan dataset training yang dihasilkan
import json
if os.path.exists("training_data/shiro_alpaca_train.json"):
    with open("training_data/shiro_alpaca_train.json", "r", encoding="utf-8") as f:
        sample = json.load(f)
    print(f"\n✓ Total data latih terkumpul: {len(sample)} pasang percakapan.")
    print("Contoh data pertama:", json.dumps(sample[0] if sample else {}, indent=2, ensure_ascii=False))
